# Soft-Routing Comparison: Shared Representation vs. Dedicated Task Models

This notebook answers the question raised for the paper's routing-failure analysis:
does replacing **hard routing** (Task 1's decision collapsed to argmax, gating whether
Task 2 ever runs) with **soft routing** (Task 1's probability blended continuously into
the final decision) change which architecture wins — the shared-encoder hierarchy or
dedicated per-task models — and does either soft-routed hierarchy still add anything
over the flat classifier?

## What is reused vs. what is new here

Reused, unmodified repo code:
- `ISIC2019HierarchicalDataset` / `build_eval_transform` (`src/data/isic2019_dataset.py`,
  `src/data/transforms.py`) — identical frozen internal-test split and preprocessing
  used everywhere else in this project.
- `build_classification_model` (`src/models/classification_backbone.py`) — builds the
  two standalone (dedicated) Task 1 / Task 2 models.
- `build_shared_three_task_model` (`src/models/shared_three_task.py`) — builds the
  shared-encoder model.
- `compute_classification_metrics` (`src/evaluation/classification_metrics.py`) — the
  same accuracy/macro-F1/confusion-matrix computation used by every other evaluation
  in this repo.

New in this notebook (does not exist in `src/` yet, because the repo currently only
implements **hard** routing in `src/evaluation/hierarchical_evaluator.py`):
- `compose_soft_routing(...)` — the soft-routing probability composition
  (documented below).
- A small probability-collection loop for the shared model that keeps Task 2's
  probabilities for *every* image (the existing `collect_shared_isic_predictions` in
  `src/evaluation/phase04_comparative_harness.py` intentionally masks Task 2's output
  to NaN wherever hard routing would have blocked it — that masking is specific to the
  hard-routing bookkeeping and is exactly what soft routing must NOT do).

## The soft-routing formula

Task 1 outputs `P1 = [P(non_malignant), P(malignant)]`.
Task 2 outputs `P2 = [P(melanoma), P(bcc), P(scc)]` (always computed, for every image,
regardless of what Task 1 said).

The final 4-class probability distribution is composed by the chain rule instead of a
hard gate:

```
P(non_malignant) = P1(non_malignant)
P(melanoma)      = P1(malignant) * P2(melanoma)
P(bcc)            = P1(malignant) * P2(bcc)
P(scc)            = P1(malignant) * P2(scc)
```

These four numbers always sum to 1 (`P1(non_malignant) + P1(malignant) * 1 = 1`), so it
is a valid probability distribution, and the predicted class is simply its `argmax` —
same decision rule as everywhere else in this project, just fed a differently-composed
probability vector. Unlike hard routing, a borderline Task 1 call (e.g. 51% malignant)
no longer fully silences or fully admits Task 2 — its opinion is always present, scaled
by how confident Task 1 was.

**Oracle soft routing** (the soft-routing analogue of the paper's existing oracle
diagnostic) replaces `P1` with the one-hot ground-truth stage-1 label, keeping Task 2's
learned probabilities unchanged — isolating how much soft routing itself, independent
of any remaining Task 1 error, leaves on the table.

## Does a dedicated + soft-routed hierarchy just collapse into the flat classifier?

No — and this notebook's results are what will let the paper state that with evidence
either way. Three structural differences remain even under soft routing:

1. **Different training objectives per stage.** The flat classifier is trained once
   with one 4-way cross-entropy loss. The dedicated hierarchy trains Task 1 as a binary
   decision and Task 2 as a 3-way decision *restricted to malignant images*, each with
   its own loss — Task 2 here uses class-balanced focal loss tuned for the rare SCC
   class (`configs/experiments/phase04_stage02_isic2019_efficientnet_b0_class_balanced_focal_loss.yaml`),
   which the flat classifier's single shared loss does not give SCC individually. A
   4-way flat loss dilutes a minority class's gradient across all three majority
   classes at once; a dedicated Task 2 head only ever has to separate the malignant
   subtypes from each other.
2. **An explicit clinical prior is encoded structurally**, not just learned implicitly:
   the algebra above *forces* `P(melanoma) + P(bcc) + P(scc) = P1(malignant)` exactly.
   The flat classifier has no such constraint — it could in principle assign
   `P(melanoma) = 0.9` while also assigning a high probability to some non-malignant
   class if its training never enforced consistency between "is it malignant" and
   "which subtype." The hierarchy's decomposition guarantees that consistency.
3. **Two independently optimizable models** (in the dedicated case) means each can use
   a different backbone, learning rate schedule, or augmentation policy if that later
   turns out to help one stage more than the other — a flat classifier is one
   monolithic decision surface.

Whether these structural differences translate into a *better* macro-F1 than the flat
classifier is exactly what this notebook measures — it is not guaranteed by the
architecture alone, which is the honest framing for the paper regardless of which way
the numbers land.

## Prerequisites — train these first (see the Setup cell for exact commands)

This notebook only evaluates and composes results; it does not train models. You need,
at minimum:
1. The shared three-task model (`scripts/train_phase03_shared_three_task.py`).
2. The standalone Task 1 model (`configs/experiments/phase03_stage01_isic2019_efficientnet_b0_cross_entropy.yaml`).
3. The standalone Task 2 model (`configs/experiments/phase04_stage02_isic2019_efficientnet_b0_class_balanced_focal_loss.yaml`).

The flat classifier (for the reference bar in the final comparison) is whatever you
already trained in `01_phase06_efficientnet_b0_flat_four_class.ipynb`.


## 0. Setup

Train the three prerequisite models from a terminal (not this notebook — these are
plain, unmodified repo scripts, not the ones with the local Windows workarounds we
applied to the flat-classifier notebooks; run them exactly as the repo intends,
adding `--device cpu` if you do not have a working CUDA setup for these runs):

```powershell
# 1. Shared three-task model (encoder + Task1/Task2/Task3 heads, one training run)
python scripts\train_phase03_shared_three_task.py --device cuda

# 2. Standalone Task 1 (malignancy gate)
python scripts\train_isic2019_baseline.py --config configs\experiments\phase03_stage01_isic2019_efficientnet_b0_cross_entropy.yaml --device cuda

# 3. Standalone Task 2 (subtype, malignant-only, class-balanced focal loss)
python scripts\train_isic2019_baseline.py --config configs\experiments\phase04_stage02_isic2019_efficientnet_b0_class_balanced_focal_loss.yaml --device cuda
```

Each prints its `best_checkpoint` path when it finishes — paste those paths into the
next cell. If you hit the same Windows `num_workers` hang described earlier in this
project, add `--config` pointing at a temporary copy of the YAML with
`loader.num_workers: 0` (see the earlier flat-classifier notebooks for the exact
pattern), since these are plain scripts and won't apply that workaround
automatically.


In [ ]:
import copyreg
import platform
import sys
from pathlib import Path
from types import MappingProxyType

import torch


# Windows creates DataLoader worker processes via "spawn", which pickles the
# Dataset object to send to each worker. This repo's dataset stores an
# index_to_class attribute as a MappingProxyType, which the standard pickle
# module cannot serialize on any platform by default. Registering a reducer
# here only teaches pickle how to serialize that one built-in type; it does
# not change any model, data, or training logic.
def _rebuild_mappingproxy(mapping):
    return MappingProxyType(mapping)


def _reduce_mappingproxy(obj):
    return _rebuild_mappingproxy, (dict(obj),)


copyreg.pickle(MappingProxyType, _reduce_mappingproxy)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"project_root={PROJECT_ROOT}")
print(f"device={DEVICE}")


In [ ]:
# Fill these in with the best_checkpoint.pt paths printed by the three training
# commands above (absolute paths, or paths relative to PROJECT_ROOT).

SHARED_CHECKPOINT_PATH = PROJECT_ROOT / "runs/phase03_shared_three_task/seed_42/best_checkpoint.pt"
TASK1_STANDALONE_CHECKPOINT_PATH = PROJECT_ROOT / "REPLACE_WITH_YOUR_TASK1_RUN/best_checkpoint.pt"
TASK2_STANDALONE_CHECKPOINT_PATH = PROJECT_ROOT / "REPLACE_WITH_YOUR_TASK2_RUN/best_checkpoint.pt"

# Optional: the flat classifier from notebook 01, used only as a reference bar in
# the final comparison. Leave as None to skip it.
FLAT_CHECKPOINT_PATH = None  # e.g. PROJECT_ROOT / "experiments/runs/full__phase06_.../best_checkpoint.pt"

for name, path in {
    "shared": SHARED_CHECKPOINT_PATH,
    "task1_standalone": TASK1_STANDALONE_CHECKPOINT_PATH,
    "task2_standalone": TASK2_STANDALONE_CHECKPOINT_PATH,
}.items():
    print(f"{name}: {path} exists={Path(path).is_file()}")


## 1. Preprocessing — one frozen internal-test loader shared by every model

Every model in this comparison (shared, standalone Task 1, standalone Task 2, flat)
takes the exact same input: one `224x224` ISIC 2019 image, deterministically
preprocessed (`resize 256 -> center-crop 224 -> ImageNet normalize`, no augmentation).
So one `flat_four_class` loader over the frozen internal-test split gives us
everything: the images to feed every model, and (by construction, the same way
`collect_shared_isic_predictions` derives it) the ground-truth Task 1 and Task 2
labels for every image.


In [ ]:
from torch.utils.data import DataLoader

from src.data.isic2019_dataset import ISIC2019HierarchicalDataset
from src.data.transforms import build_eval_transform
from src.utils.reproducibility import make_generator, seed_worker

SPLIT_MANIFEST = PROJECT_ROOT / "data/manifests/isic2019_train_val_test_split_seed42.csv"
SEED = 42
BATCH_SIZE = 64
NUM_WORKERS = 0  # kept at 0 for the same local-Windows-DataLoader reasons as the other notebooks

eval_transform = build_eval_transform()
internal_test_dataset = ISIC2019HierarchicalDataset(
    SPLIT_MANIFEST, PROJECT_ROOT, "internal_test", "flat_four_class", eval_transform
)
internal_test_loader = DataLoader(
    internal_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
    drop_last=False,
    worker_init_fn=seed_worker,
    generator=make_generator(SEED),
)
print(f"internal_test samples: {len(internal_test_dataset)}")


## 2. Model definition

Build all three models exactly as their own training configs define them, then load
each frozen checkpoint. `pretrained="none"` is used everywhere here because we are
about to overwrite every weight with the trained checkpoint anyway — this only skips
a redundant ImageNet download, it does not change which weights end up loaded.


In [ ]:
from src.models.classification_backbone import build_classification_model
from src.models.shared_three_task import build_shared_three_task_model

# --- Shared three-task model ---
shared_payload = torch.load(SHARED_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
shared_architecture = shared_payload["model_metadata"]["architecture"]
shared_model = build_shared_three_task_model(shared_architecture, pretrained="none")
shared_model.load_state_dict(shared_payload["model_state_dict"], strict=True)
shared_model.to(DEVICE).eval()
print(f"shared model: architecture={shared_architecture} epoch={shared_payload['epoch']} "
      f"task1_val_macro_f1={shared_payload['task1_val_macro_f1']:.4f} "
      f"task2_val_macro_f1={shared_payload['task2_val_macro_f1']:.4f}")

# --- Standalone Task 1 (non_malignant vs malignant) ---
task1_payload = torch.load(TASK1_STANDALONE_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
task1_config = task1_payload["config"]
task1_model = build_classification_model(
    task1_config["model"]["architecture"],
    task1_config["model"]["number_of_classes"],
    pretrained="none",
    dropout_probability=task1_config["model"].get("dropout_probability", 0.2),
)
task1_model.load_state_dict(task1_payload["model_state_dict"], strict=True)
task1_model.to(DEVICE).eval()
task1_class_names = list(task1_payload["class_names"])
print(f"standalone task1: architecture={task1_config['model']['architecture']} "
      f"epoch={task1_payload['epoch']} class_names={task1_class_names}")

# --- Standalone Task 2 (melanoma / bcc / scc) ---
task2_payload = torch.load(TASK2_STANDALONE_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
task2_config = task2_payload["config"]
task2_model = build_classification_model(
    task2_config["model"]["architecture"],
    task2_config["model"]["number_of_classes"],
    pretrained="none",
    dropout_probability=task2_config["model"].get("dropout_probability", 0.2),
)
task2_model.load_state_dict(task2_payload["model_state_dict"], strict=True)
task2_model.to(DEVICE).eval()
task2_class_names = list(task2_payload["class_names"])
print(f"standalone task2: architecture={task2_config['model']['architecture']} "
      f"epoch={task2_payload['epoch']} class_names={task2_class_names}")

assert task1_class_names == ["non_malignant", "malignant"], task1_class_names
assert task2_class_names == ["melanoma", "bcc", "scc"], task2_class_names


## 3. Collect raw probabilities from both systems

For both the shared model and the dedicated (standalone) pair, we run one full pass
over the internal-test loader and record **every** image's Task 1 probability and
Task 2 probability — unmasked, unlike the hard-routing bookkeeping elsewhere in this
repo, because soft routing needs Task 2's opinion on every image, not just the ones
Task 1 would have hard-routed to it.

Also recovers the true stage-1 / stage-2 labels the same way
`collect_shared_isic_predictions` (`src/evaluation/phase04_comparative_harness.py`)
already does, so the ground truth used here is identical to the rest of the project.


In [ ]:
import numpy as np

def collect_all_probabilities(shared_model, task1_model, task2_model, dataloader, device):
    """One pass over the loader collecting flat targets plus raw Task1/Task2
    probabilities from BOTH the shared model and the two standalone models.

    This does not exist in src/ because the repo's existing shared-model collector
    (collect_shared_isic_predictions) intentionally masks Task 2 to NaN wherever hard
    routing would have blocked it. Soft routing needs Task 2's probability for every
    image, so this keeps it unmasked for both systems while reusing every model's
    forward() unchanged.
    """
    image_ids = []
    flat_targets = []
    shared_p1, shared_p2 = [], []
    dedicated_p1, dedicated_p2 = [], []

    with torch.inference_mode():
        for batch in dataloader:
            images = batch["image"].to(device, non_blocking=True)
            targets = batch["target"]
            count = images.shape[0]

            shared_out = shared_model(images)
            shared_p1_batch = torch.softmax(shared_out["task1"].float(), dim=1).cpu()
            shared_p2_batch = torch.softmax(shared_out["task2"].float(), dim=1).cpu()

            task1_logits = task1_model(images)
            task2_logits = task2_model(images)
            dedicated_p1_batch = torch.softmax(task1_logits.float(), dim=1).cpu()
            dedicated_p2_batch = torch.softmax(task2_logits.float(), dim=1).cpu()

            image_ids.extend(batch["image_id"])
            flat_targets.append(targets)
            shared_p1.append(shared_p1_batch)
            shared_p2.append(shared_p2_batch)
            dedicated_p1.append(dedicated_p1_batch)
            dedicated_p2.append(dedicated_p2_batch)

    return {
        "image_ids": image_ids,
        "flat_targets": torch.cat(flat_targets).numpy(),
        "shared_p1": torch.cat(shared_p1).numpy(),
        "shared_p2": torch.cat(shared_p2).numpy(),
        "dedicated_p1": torch.cat(dedicated_p1).numpy(),
        "dedicated_p2": torch.cat(dedicated_p2).numpy(),
    }


collected = collect_all_probabilities(shared_model, task1_model, task2_model, internal_test_loader, DEVICE)

# Ground truth, derived exactly as collect_shared_isic_predictions does:
# flat target 0 = non_malignant; 1/2/3 = melanoma/bcc/scc (already malignant-only offsets).
flat_targets = collected["flat_targets"]
stage1_targets = (flat_targets != 0).astype(np.int64)          # 0=non_malignant, 1=malignant
stage2_targets = np.where(flat_targets == 0, -1, flat_targets - 1)  # -1 where not malignant

print(f"collected probabilities for {len(collected['image_ids'])} images")
print(f"true malignant count: {int(stage1_targets.sum())} / {len(stage1_targets)}")


## 4. Soft-routing composition

This is the one genuinely new piece of logic in this notebook — the soft-routing
formula described in the introduction above, applied identically to both systems'
probabilities so the comparison is fair. `oracle=True` substitutes the ground-truth
stage-1 label for `p1`, which is the soft-routing analogue of the paper's existing
oracle-routing diagnostic.


In [ ]:
FINAL_CLASS_NAMES = ["non_malignant", "melanoma", "bcc", "scc"]


def compose_soft_routing(p1, p2, *, stage1_targets=None, oracle=False):
    """Compose Task1 (2-class) and Task2 (3-class) probabilities into one 4-class
    distribution via the hierarchy's chain rule, instead of hard-gating on argmax(p1).

    p1: [N, 2] array, columns = (non_malignant, malignant)
    p2: [N, 3] array, columns = (melanoma, bcc, scc)
    oracle=True replaces p1 with the one-hot ground-truth stage-1 label (soft-routing's
    analogue of the paper's oracle-routing diagnostic), leaving p2 untouched.
    """
    if oracle:
        if stage1_targets is None:
            raise ValueError("oracle routing requires stage1_targets.")
        p_malignant = stage1_targets.astype(np.float64)
    else:
        p_malignant = p1[:, 1]

    p_non_malignant = 1.0 - p_malignant
    composed = np.stack(
        [
            p_non_malignant,
            p_malignant * p2[:, 0],
            p_malignant * p2[:, 1],
            p_malignant * p2[:, 2],
        ],
        axis=1,
    )
    # Should already sum to 1 by construction; renormalize only to absorb floating-point drift.
    composed = composed / composed.sum(axis=1, keepdims=True)
    return composed


def predicted_and_metrics(probabilities, targets):
    from src.evaluation.classification_metrics import compute_classification_metrics

    predictions = probabilities.argmax(axis=1)
    return predictions, compute_classification_metrics(targets, predictions, FINAL_CLASS_NAMES)


## 5. Evaluation — Condition A: shared representation, soft routing

This is the notebook's answer to "get an evaluation using the current shared
representation learning [under soft routing]."


In [ ]:
shared_soft_predicted_probs = compose_soft_routing(collected["shared_p1"], collected["shared_p2"])
shared_soft_predicted_preds, shared_soft_predicted_metrics = predicted_and_metrics(
    shared_soft_predicted_probs, flat_targets
)

shared_soft_oracle_probs = compose_soft_routing(
    collected["shared_p1"], collected["shared_p2"], stage1_targets=stage1_targets, oracle=True
)
shared_soft_oracle_preds, shared_soft_oracle_metrics = predicted_and_metrics(
    shared_soft_oracle_probs, flat_targets
)

print(f"[shared | soft | predicted routing] macro_f1={shared_soft_predicted_metrics['macro_f1']:.4f} "
      f"balanced_accuracy={shared_soft_predicted_metrics['balanced_accuracy']:.4f}")
print(f"[shared | soft | oracle routing]    macro_f1={shared_soft_oracle_metrics['macro_f1']:.4f} "
      f"balanced_accuracy={shared_soft_oracle_metrics['balanced_accuracy']:.4f}")
print(f"routing_loss_macro_f1 (soft) = {shared_soft_oracle_metrics['macro_f1'] - shared_soft_predicted_metrics['macro_f1']:.4f}")


## 6. Evaluation — Condition B: dedicated (standalone) models, soft routing

This is the notebook's answer to "get an evaluation using the dedicated model based
system [under the same soft routing]." Same formula, same ground truth, same metric
function — the only difference from Condition A is which two models produced `p1`
and `p2`.


In [ ]:
dedicated_soft_predicted_probs = compose_soft_routing(collected["dedicated_p1"], collected["dedicated_p2"])
dedicated_soft_predicted_preds, dedicated_soft_predicted_metrics = predicted_and_metrics(
    dedicated_soft_predicted_probs, flat_targets
)

dedicated_soft_oracle_probs = compose_soft_routing(
    collected["dedicated_p1"], collected["dedicated_p2"], stage1_targets=stage1_targets, oracle=True
)
dedicated_soft_oracle_preds, dedicated_soft_oracle_metrics = predicted_and_metrics(
    dedicated_soft_oracle_probs, flat_targets
)

print(f"[dedicated | soft | predicted routing] macro_f1={dedicated_soft_predicted_metrics['macro_f1']:.4f} "
      f"balanced_accuracy={dedicated_soft_predicted_metrics['balanced_accuracy']:.4f}")
print(f"[dedicated | soft | oracle routing]    macro_f1={dedicated_soft_oracle_metrics['macro_f1']:.4f} "
      f"balanced_accuracy={dedicated_soft_oracle_metrics['balanced_accuracy']:.4f}")
print(f"routing_loss_macro_f1 (soft) = {dedicated_soft_oracle_metrics['macro_f1'] - dedicated_soft_predicted_metrics['macro_f1']:.4f}")


## 7. Optional reference row — the flat classifier

Only runs if `FLAT_CHECKPOINT_PATH` was set above. Uses the same
`build_classification_model` + checkpoint-loading pattern as every other notebook in
this project, and the same frozen internal-test loader as everything above, so it is
directly comparable.


In [ ]:
flat_metrics = None
if FLAT_CHECKPOINT_PATH is not None:
    flat_payload = torch.load(FLAT_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
    flat_config = flat_payload["config"]
    flat_model = build_classification_model(
        flat_config["model"]["architecture"],
        flat_config["model"]["number_of_classes"],
        pretrained="none",
        dropout_probability=flat_config["model"].get("dropout_probability", 0.2),
    )
    flat_model.load_state_dict(flat_payload["model_state_dict"], strict=True)
    flat_model.to(DEVICE).eval()

    flat_probs_chunks = []
    with torch.inference_mode():
        for batch in internal_test_loader:
            images = batch["image"].to(DEVICE, non_blocking=True)
            logits = flat_model(images)
            flat_probs_chunks.append(torch.softmax(logits.float(), dim=1).cpu())
    flat_probs = torch.cat(flat_probs_chunks).numpy()
    _, flat_metrics = predicted_and_metrics(flat_probs, flat_targets)
    print(f"[flat classifier] macro_f1={flat_metrics['macro_f1']:.4f} "
          f"balanced_accuracy={flat_metrics['balanced_accuracy']:.4f}")
else:
    print("FLAT_CHECKPOINT_PATH not set; skipping the flat-classifier reference row.")


## 8. Comparison table and chart for the paper

Collects every condition run above into one table, alongside the paper's existing
hard-routing numbers (typed in as fixed reference values from
`reports/phase04_controlled_comparative/final_internal_test/final_routing_analysis.json`
and the paper text) so the new soft-routing results sit next to what is already
published.


In [ ]:
import pandas as pd

# Existing hard-routing figures already reported in the paper / phase04 evidence,
# included here only as fixed reference points -- not recomputed by this notebook.
EXISTING_HARD_ROUTING_REFERENCE = {
    "flat classifier (paper, hard n/a)": 0.6192,
    "shared | hard | predicted routing (paper)": 0.5686,
    "shared | hard | oracle routing (paper)": 0.7770,
}

rows = [
    {"system": "shared", "routing": "soft", "condition": "predicted", "macro_f1": shared_soft_predicted_metrics["macro_f1"], "balanced_accuracy": shared_soft_predicted_metrics["balanced_accuracy"]},
    {"system": "shared", "routing": "soft", "condition": "oracle", "macro_f1": shared_soft_oracle_metrics["macro_f1"], "balanced_accuracy": shared_soft_oracle_metrics["balanced_accuracy"]},
    {"system": "dedicated", "routing": "soft", "condition": "predicted", "macro_f1": dedicated_soft_predicted_metrics["macro_f1"], "balanced_accuracy": dedicated_soft_predicted_metrics["balanced_accuracy"]},
    {"system": "dedicated", "routing": "soft", "condition": "oracle", "macro_f1": dedicated_soft_oracle_metrics["macro_f1"], "balanced_accuracy": dedicated_soft_oracle_metrics["balanced_accuracy"]},
]
if flat_metrics is not None:
    rows.append({"system": "flat", "routing": "n/a", "condition": "n/a", "macro_f1": flat_metrics["macro_f1"], "balanced_accuracy": flat_metrics["balanced_accuracy"]})

comparison_table = pd.DataFrame(rows)
print("New soft-routing results (this run):")
display(comparison_table)

print("\nExisting hard-routing reference values (paper):")
for label, value in EXISTING_HARD_ROUTING_REFERENCE.items():
    print(f"  {label}: {value:.4f}")


In [ ]:
import matplotlib.pyplot as plt

labels = list(EXISTING_HARD_ROUTING_REFERENCE.keys()) + [
    f"{row['system']} | {row['routing']} | {row['condition']}" for row in rows
]
values = list(EXISTING_HARD_ROUTING_REFERENCE.values()) + [row["macro_f1"] for row in rows]
colors = ["#9e9e9e"] * len(EXISTING_HARD_ROUTING_REFERENCE) + ["#1f77b4"] * len(rows)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(labels, values, color=colors)
ax.set_ylabel("Macro-F1 (internal test)")
ax.set_title("Flat vs. shared vs. dedicated, hard vs. soft routing")
ax.set_ylim(0, 1)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## 9. Save outputs

Writes everything to a fresh, clearly separate directory so it never overwrites the
frozen `reports/phase04_controlled_comparative/` evidence this analysis builds on top
of.


In [ ]:
import json
from datetime import datetime, timezone

output_dir = PROJECT_ROOT / "experiments" / "evaluations" / "soft_routing_comparison"
output_dir.mkdir(parents=True, exist_ok=True)

comparison_table.to_csv(output_dir / "soft_routing_comparison_table.csv", index=False)

summary = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "internal_test_sample_count": len(collected["image_ids"]),
    "shared_checkpoint_path": str(SHARED_CHECKPOINT_PATH),
    "task1_standalone_checkpoint_path": str(TASK1_STANDALONE_CHECKPOINT_PATH),
    "task2_standalone_checkpoint_path": str(TASK2_STANDALONE_CHECKPOINT_PATH),
    "flat_checkpoint_path": str(FLAT_CHECKPOINT_PATH) if FLAT_CHECKPOINT_PATH else None,
    "existing_hard_routing_reference": EXISTING_HARD_ROUTING_REFERENCE,
    "results": {
        "shared_soft_predicted": shared_soft_predicted_metrics,
        "shared_soft_oracle": shared_soft_oracle_metrics,
        "dedicated_soft_predicted": dedicated_soft_predicted_metrics,
        "dedicated_soft_oracle": dedicated_soft_oracle_metrics,
        "flat": flat_metrics,
    },
}
with open(output_dir / "soft_routing_comparison_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, sort_keys=True)

print(f"wrote {output_dir / 'soft_routing_comparison_table.csv'}")
print(f"wrote {output_dir / 'soft_routing_comparison_summary.json'}")


## 10. Reading the results for the paper

Once the cells above have run, look at these specific comparisons:

1. **Shared vs. dedicated, both soft-routed, predicted condition** — this is the
   direct answer to "is a dedicated model system more plausible." If
   `dedicated_soft_predicted_metrics["macro_f1"] > shared_soft_predicted_metrics["macro_f1"]`,
   that extends your paper's existing Section IV-D finding (dedicated Task1/Task2
   beat their shared counterparts individually) to the full deployed 4-class
   endpoint, now under a fairer (soft) routing rule.
2. **Soft vs. hard routing, same shared model** — compare
   `shared_soft_predicted_metrics["macro_f1"]` against the paper's existing hard
   routing figure (0.5686). If soft routing recovers meaningfully more than zero
   while still trailing the oracle ceiling (0.7770), that is direct evidence the
   paper's proposed fix (soft/uncertainty-aware gating) works, and how much of the
   0.2084 routing-associated gap it closes.
3. **Best hierarchical condition vs. flat (0.6192)** — this settles the redundancy
   question empirically: if the best soft-routed hierarchical macro-F1 still falls
   short of the flat classifier, that is evidence the flat model remains the stronger
   *deployed* choice even after fixing routing, and the value of the hierarchy is
   diagnostic/interpretive (per-stage class-imbalance handling, explicit clinical
   structure) rather than a raw macro-F1 win. If it matches or exceeds flat, that is
   evidence soft, dedicated routing genuinely recovers the hierarchy's structural
   advantage.
4. **Routing-associated gap, soft vs. hard** — report
   `oracle_macro_f1 - predicted_macro_f1` for both hard (existing: 0.2084) and soft
   (computed above) routing on the same shared model. A smaller gap under soft
   routing is itself evidence for the mechanism, independent of whether the absolute
   macro-F1 beats the flat classifier.
